In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [2]:
spark = SparkSession.builder \
    .appName("credit-risk-lakehouse-proposta-credito") \
    .master("spark://spark-master:7077") \
    .getOrCreate()

spark

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/24 02:02:58 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [ ]:
BASE_PATH = "/app/data"

In [4]:
df_cliente = spark.read.parquet(f"{BASE_PATH}/bronze/cliente")

df_cliente.count()

10000

In [5]:
NUM_PROPOSTAS = 30_000
NUM_CLIENTES = df_cliente.count()

df_proposta_credito = spark.range(1, NUM_PROPOSTAS + 1) \
    .withColumnRenamed("id", "proposta_id") \
    .withColumn(
        "cliente_id",
        floor(rand() * NUM_CLIENTES + 1).cast("long")
    ) \
    .withColumn(
        "produto",
        element_at(
            array(
                lit("credito_pessoal"),
                lit("financiamento_veiculo"),
                lit("cartao_credito"),
                lit("emprestimo_consignado"),
                lit("capital_giro")
            ),
            floor(rand() * 5 + 1).cast("int")
        )
    ) \
    .withColumn(
        "valor_solicitado",
        round(rand() * 100000 + 1000, 2)
    ) \
    .withColumn(
        "prazo_meses",
        element_at(
            array(
                lit(6),
                lit(12),
                lit(18),
                lit(24),
                lit(36),
                lit(48),
                lit(60)
            ),
            floor(rand() * 7 + 1).cast("int")
        )
    ) \
    .withColumn(
        "data_proposta",
        date_sub(
            current_date(),
            floor(rand() * 1460).cast("int")
        )
    ) \
    .withColumn(
        "status_proposta",
        element_at(
            array(
                lit("aprovada"),
                lit("aprovada"),
                lit("aprovada"),
                lit("reprovada"),
                lit("cancelada")
            ),
            floor(rand() * 5 + 1).cast("int")
        )
    )

In [9]:
df_proposta_credito.write \
    .mode("overwrite") \
    .parquet(
        f"{BASE_PATH}/bronze/proposta_credito"
    )

In [10]:
spark.stop()